# Jaccard Coeff

## Load Data

In [7]:
import pandas as pd
import numpy as np

movies = pd.read_csv("/content/movies.csv")
movies.head()


,index,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,cast,crew,director
0,0,237000000,Action Adventure Fantasy Science Fiction,http://www.avatarmovie.com/,19995,culture clash future space war space colony so...,en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,Sam Worthington Zoe Saldana Sigourney Weaver S...,"[{'name': 'Stephen E. Rivkin', 'gender': 0, 'd...",James Cameron
1,1,300000000,Adventure Fantasy Action,http://disney.go.com/disneypictures/pirates/,285,ocean drug abuse exotic island east india trad...,en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,Johnny Depp Orlando Bloom Keira Knightley Stel...,"[{'name': 'Dariusz Wolski', 'gender': 2, 'depa...",Gore Verbinski
2,2,245000000,Action Adventure Crime,http://www.sonypictures.com/movies/spectre/,206647,spy based on novel secret agent sequel mi6,en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,Daniel Craig Christoph Waltz L\u00e9a Seydoux ...,"[{'name': 'Thomas Newman', 'gender': 2, 'depar...",Sam Mendes
3,3,250000000,Action Crime Drama Thriller,http://www.thedarkknightrises.com/,49026,dc comics crime fighter terrorist secret ident...,en,The Dark Knight Rises,Following the death of District Attorney Harve...,112.312950,...,165.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,The Legend Ends,The Dark Knight Rises,7.6,9106,Christian Bale Michael Caine Gary Oldman Anne ...,"[{'name': 'Hans Zimmer', 'gender': 2, 'departm...",Christopher Nolan
4,4,260000000,Action Adventure Science Fiction,http://movies.disney.com/john-carter,49529,based on novel mars medallion space travel pri...,en,John Carter,"John Carter is a war-weary, former military ca...",43.926995,...,132.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"Lost in our world, found in another.",John Carter,6.1,2124,Taylor Kitsch Lynn Collins Samantha Morton Wil...,"[{'name': 'Andrew Stanton', 'gender': 2, 'depa...",Andrew Stanton


## Clean Genres

In [8]:
movies['genres'] = movies['genres'].replace('(no genres listed)', '')
movies['genres_list'] = movies['genres'].str.split()

## Multi-Hot Encode Genres

In [10]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

# Replace any NaN values in 'genres_list' with empty lists
cleaned_genres_list = movies['genres_list'].apply(lambda x: x if isinstance(x, list) else [])

genre_matrix = mlb.fit_transform(cleaned_genres_list)

print(genre_matrix.shape)   # (num_movies, num_genres)
print(mlb.classes_)         # list of genres

(4803, 22)
['Action' 'Adventure' 'Animation' 'Comedy' 'Crime' 'Documentary' 'Drama'
 'Family' 'Fantasy' 'Fiction' 'Foreign' 'History' 'Horror' 'Movie' 'Music'
 'Mystery' 'Romance' 'Science' 'TV' 'Thriller' 'War' 'Western']


## Jaccard Similarity Matrix

In [11]:
from sklearn.metrics import pairwise_distances

# Jaccard distance → convert to similarity
jaccard_similarity = 1 - pairwise_distances(
    genre_matrix,
    metric="jaccard"
)


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/pairwise.py:2466: DataConversionWarning: Data was converted to boolean for metric jaccard
  warnings.warn(msg, DataConversionWarning)


## Recommendation Function

In [12]:
def recommend_by_jaccard(movie_title, top_n=5):
    if movie_title not in movies['title'].values:
        return "Movie not found"

    idx = movies[movies['title'] == movie_title].index[0]

    sim_scores = list(enumerate(jaccard_similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # skip itself
    sim_scores = sim_scores[1:top_n+1]
    movie_indices = [i[0] for i in sim_scores]

    return movies[['title', 'genres']].iloc[movie_indices]


## Test

In [15]:
recommend_by_jaccard("Avatar", top_n=10)


,title,genres
10,Superman Returns,Adventure Fantasy Action Science Fiction
14,Man of Steel,Action Adventure Fantasy Science Fiction
46,X-Men: Days of Future Past,Action Adventure Fantasy Science Fiction
61,Jupiter Ascending,Science Fiction Fantasy Action Adventure
232,The Wolverine,Action Science Fiction Adventure Fantasy
813,Superman,Action Adventure Fantasy Science Fiction
870,Superman II,Action Adventure Fantasy Science Fiction
3494,Beastmaster 2: Through the Portal of Time,Action Adventure Fantasy Science Fiction
72,Suicide Squad,Action Adventure Crime Fantasy Science Fiction
168,Final Fantasy: The Spirits Within,Adventure Action Animation Fantasy Science Fic...


# RNN

# AutoEncoder